# 🎬 Animación Orbital FAE — 60 minutos de pases satelitales

Visualiza el **movimiento en tiempo real** de satélites sobre Ecuador.
Genera un video MP4 con análisis de ventanas de comunicación.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.patches as mpatches
from matplotlib.collections import LineCollection
import math
from datetime import datetime, timezone, timedelta
import warnings
warnings.filterwarnings('ignore')

print('✅ Librerías de animación listas')

In [ ]:
# ── REUTILIZAR DATOS DEL NOTEBOOK ANTERIOR ──
# Si ejecutaste FAE_Matplotlib_Grafos antes, estos ya existen.
# Si no, descomenta esto:

import random
random.seed(42)

# Constantes orbitales
RE = 6371.0
D2R = math.pi / 180

# ── Satélites ──
configs = [
    ('GPS',      '#4FC3F7', 20200, 55.0,  718.0, 10),
    ('Galileo',  '#81C784', 23222, 56.0,  844.0,  8),
    ('Starlink', '#FFB74D',   550, 53.0,   95.5, 12),
    ('GOES',     '#CE93D8', 35786,  0.0, 1436.0,  3),
]
GEO_LONS = [-75.2, -135.0, -60.0]

satelites = []
sid = 0
for cons, color, alt, inc, period, n in configs:
    for i in range(n):
        raan = GEO_LONS[i%3] if inc==0 else (i*360/n + random.uniform(-10,10))%360
        m0   = 0.0 if inc==0 else random.uniform(0,360)
        satelites.append({
            'id':sid,'nombre':f'{cons}-{i+1}','constelacion':cons,
            'color':color,'alt_km':alt,'inc_deg':inc,
            'period_min':period,'raan':raan,'m0':m0,'geo':inc==0.0
        })
        sid += 1

# ── Estaciones ──
estaciones = [
    {'nombre':'Quito FAE',  'lat':-0.22,'lon':-78.51,'alt':2.85},
    {'nombre':'Guayaquil',  'lat':-2.17,'lon':-79.92,'alt':0.004},
    {'nombre':'Manta',      'lat':-0.97,'lon':-80.71,'alt':0.018},
    {'nombre':'Lago Agrio', 'lat': 0.08,'lon':-76.88,'alt':0.297},
    {'nombre':'Galapagos',  'lat':-0.90,'lon':-89.60,'alt':0.006},
]

print(f'✅ Satélites: {len(satelites)} | Estaciones: {len(estaciones)}')

In [ ]:
# ── FUNCIONES ORBITALES ──
def gmst(d):
    J = d.timestamp()/86400 + 2440587.5
    T = (J-2451545)/36525
    g = 280.46061837 + 360.98564736629*(J-2451545) + T*T*0.000387933
    return g % 360

def sat_ecef(s, d):
    if s['geo']:
        l = s['raan']*D2R
        r = RE+s['alt_km']
        return np.array([r*math.cos(l), r*math.sin(l), 0])
    n  = 2*math.pi/(s['period_min']*60)
    t  = d.timestamp()
    M  = (s['m0']*D2R + n*t) % (2*math.pi)
    r  = RE+s['alt_km']
    inc= s['inc_deg']*D2R
    ra = (s['raan'] + gmst(d)*0.985)*D2R
    xo,yo = r*math.cos(M), r*math.sin(M)
    return np.array([
        math.cos(ra)*xo - math.sin(ra)*math.cos(inc)*yo,
        math.sin(ra)*xo + math.cos(ra)*math.cos(inc)*yo,
        math.sin(inc)*yo
    ])

def est_ecef(e):
    la,lo,r = e['lat']*D2R, e['lon']*D2R, RE+e['alt']
    return np.array([r*math.cos(la)*math.cos(lo),
                     r*math.cos(la)*math.sin(lo),
                     r*math.sin(la)])

def elevacion(sp, ep):
    d = sp-ep
    dist = np.linalg.norm(d)
    if dist<1: return -90
    n = ep/np.linalg.norm(ep)
    return math.asin(max(-1,min(1,np.dot(d/dist,n))))/D2R

def ecef_ll(p):
    r = np.linalg.norm(p)
    return math.asin(p[2]/r)/D2R, math.atan2(p[1],p[0])/D2R

print('✅ Funciones orbitales definidas')

In [ ]:
# ── GENERAR DATOS PARA ANIMACIÓN ──
# Simularemos 60 minutos con 1 frame por minuto (60 fps = 1 segundo de video real)

ahora = datetime.now(timezone.utc)
DURACION_MINUTOS = 60
PASO_MINUTOS = 1
EL_MIN = 10  # Elevación mínima para visibilidad

# Generar tiempos
tiempos = [ahora + timedelta(minutes=i) for i in range(0, DURACION_MINUTOS, PASO_MINUTOS)]
print(f'📅 Rango de simulación: {tiempos[0].strftime("%H:%M:%S")} → {tiempos[-1].strftime("%H:%M:%S")} UTC')
print(f'📊 Frames: {len(tiempos)} (1 por minuto)')

# Pre-calcular posiciones para todos los satélites en todos los tiempos
print('\n🔄 Calculando órbitas...')
datos_animacion = []

for frame_idx, t in enumerate(tiempos):
    frame_data = {
        'tiempo': t,
        'satelites': {},
        'visibilidad': []
    }
    
    for s in satelites:
        sp = sat_ecef(s, t)
        lat, lon = ecef_ll(sp)
        frame_data['satelites'][s['nombre']] = {
            'ecef': sp,
            'lat': lat,
            'lon': lon,
            'alt_km': s['alt_km'],
            'constelacion': s['constelacion'],
            'color': s['color']
        }
    
    # Calcular visibilidad desde cada estación
    for s in satelites:
        sp = sat_ecef(s, t)
        for e in estaciones:
            ep = est_ecef(e)
            el = elevacion(sp, ep)
            if el >= EL_MIN:
                frame_data['visibilidad'].append({
                    'sat_nombre': s['nombre'],
                    'est_nombre': e['nombre'],
                    'elevacion': round(el, 1),
                    'constelacion': s['constelacion']
                })
    
    datos_animacion.append(frame_data)
    if (frame_idx + 1) % 20 == 0:
        print(f'  ✓ Frame {frame_idx+1}/{len(tiempos)}')

print(f'✅ {len(datos_animacion)} frames pre-calculados')

In [ ]:
# ── CREAR FIGURA Y CONFIGURACIÓN PARA ANIMACIÓN ──
fig, ax = plt.subplots(figsize=(14, 14), facecolor='#060d1a')
ax.set_facecolor('#060d1a')
ax.set_aspect('equal')

RE_vis = 1.0
scale = RE_vis / RE

COLS = {'GPS': '#4FC3F7', 'Galileo': '#81C784', 'Starlink': '#FFB74D', 'GOES': '#CE93D8'}

# Elementos que persisten entre frames
orbitas_paths = {}  # Guardar historiales de satélites

print('✅ Figura de animación creada')

In [ ]:
# ── FUNCIÓN DE ANIMACIÓN ──
def animate(frame_idx):
    ax.clear()
    ax.set_facecolor('#060d1a')
    ax.set_aspect('equal')
    
    frame_data = datos_animacion[frame_idx]
    t_actual = frame_data['tiempo']
    
    # ── 1. ÓRBITAS DE REFERENCIA ──
    orbitas = [
        ('Starlink', '#FFB74D', 550),
        ('GPS', '#4FC3F7', 20200),
        ('Galileo', '#81C784', 23222),
        ('GOES', '#CE93D8', 35786)
    ]
    
    for nombre, color, alt in orbitas:
        r = (RE + alt) * scale
        circ = plt.Circle((0, 0), r, fill=False, color=color, lw=0.8, ls='--', alpha=0.25)
        ax.add_patch(circ)
    
    # ── 2. TIERRA ──
    for dr, a in zip([0.15, 0.10, 0.05], [0.05, 0.10, 0.18]):
        ax.add_patch(plt.Circle((0, 0), RE_vis + dr, color='#38bdf8', alpha=a, zorder=4))
    
    tierra = plt.Circle((0, 0), RE_vis, color='#0f172a', zorder=5)
    ax.add_patch(tierra)
    
    # Red geodésica
    theta = np.linspace(0, 2 * np.pi, 200)
    for r_grid in [0.33, 0.66]:
        ax.plot(r_grid * np.cos(theta), r_grid * np.sin(theta), 
                color='#1e3a8a', lw=0.4, alpha=0.3, zorder=6)
    
    for ang_deg in range(0, 360, 45):
        rad = math.radians(ang_deg)
        ax.plot([0, RE_vis * np.cos(rad)], [0, RE_vis * np.sin(rad)], 
                color='#1e3a8a', lw=0.4, alpha=0.3, zorder=6)
    
    # Línea ecuatorial
    ax.plot(RE_vis * np.cos(theta), RE_vis * 0.02 * np.sin(theta),
            color='#FFD700', lw=1.0, ls='--', alpha=0.6, zorder=8)
    
    # ── 3. SATÉLITES EN ÓRBITA ──
    for sat_nombre, sat_data in frame_data['satelites'].items():
        # Calcular si está visible desde alguna estación
        vis_now = any(v['sat_nombre'] == sat_nombre for v in frame_data['visibilidad'])
        
        r_vis = (RE + sat_data['alt_km']) * scale
        ang = sat_data['lon'] * math.pi / 180 - math.pi / 2
        px, py = r_vis * math.cos(ang), r_vis * math.sin(ang)
        
        sz = 100 if vis_now else 35
        alpha = 1.0 if vis_now else 0.25
        color = COLS.get(sat_data['constelacion'], '#FFFFFF')
        
        if vis_now:
            # Halo para satélites visibles
            ax.scatter(px, py, s=sz*4, color=color, alpha=0.12, zorder=9)
            # Línea de visión
            ax.plot([0, px], [0, py], color=color, alpha=0.2, lw=0.7, ls=':', zorder=3)
        
        ax.scatter(px, py, s=sz, color=color,
                   edgecolors='white' if vis_now else '#333333',
                   lw=1.0, alpha=alpha, zorder=10)
    
    # ── 4. ESTACIONES ──
    for e in estaciones:
        ang = e['lon'] * math.pi / 180 - math.pi / 2
        px_est, py_est = RE_vis * math.cos(ang), RE_vis * math.sin(ang)
        
        # Contar satélites visibles desde esta estación
        n_vis = sum(1 for v in frame_data['visibilidad'] if v['est_nombre'] == e['nombre'])
        
        if n_vis > 0:
            ax.scatter(px_est, py_est, s=500, color='#FFD700', alpha=0.15, zorder=12)
        
        ax.scatter(px_est, py_est, s=200, color='#FFD700', marker='*',
                   edgecolors='#060d1a', lw=1.0, alpha=0.9 if n_vis > 0 else 0.4, zorder=13)
        
        # Etiqueta con contador
        offset_r = 1.28
        tx, ty = RE_vis * offset_r * math.cos(ang), RE_vis * offset_r * math.sin(ang)
        
        ax.text(tx, ty, f"{e['nombre'].replace('FAE', '').strip()}\n{n_vis} sat",
                color='#FFD700', fontsize=7.5, fontweight='bold', ha='center', va='center',
                bbox=dict(boxstyle='round,pad=0.25', facecolor='#060d1a', 
                         edgecolor='#FFD700' if n_vis > 0 else '#334155', alpha=0.8, lw=0.5),
                zorder=14)
    
    # ── 5. LEYENDA DINÁMICA ──
    patches = [mpatches.Patch(color=c, label=cn) for cn, c in COLS.items()]
    patches.append(mpatches.Patch(color='#FFD700', label='Estaciones FAE ⭐'))
    
    ax.legend(handles=patches, loc='lower right', facecolor='#0f172a',
              edgecolor='#334155', labelcolor='white', fontsize=8, framealpha=0.9)
    
    # ── 6. TÍTULOS E INFORMACIÓN TEMPORAL ──
    maxr = (RE + 35786) * scale * 1.15
    ax.set_xlim(-maxr, maxr)
    ax.set_ylim(-maxr, maxr)
    ax.axis('off')
    
    ax.text(0, 1.18, '🎬 Animación Orbital FAE — Simulación en Tiempo Real',
            color='#38BDF8', fontsize=13, fontweight='bold', ha='center', zorder=100)
    
    # Información del frame
    tiempo_str = t_actual.strftime('%H:%M:%S UTC')
    num_conexiones = len(frame_data['visibilidad'])
    num_satelites = len(set(v['sat_nombre'] for v in frame_data['visibilidad']))
    
    ax.text(0, 1.10, f'{tiempo_str}  |  Conexiones: {num_conexiones}  |  Satélites visibles: {num_satelites}',
            color='white', fontsize=10, ha='center', fontweight='bold', zorder=100)
    
    # Progreso de simulación
    progreso = (frame_idx + 1) / len(datos_animacion) * 100
    ax.text(0, -1.18, f'Progreso: [{"█"*int(progreso/5)}{"░"*(20-int(progreso/5))}] {progreso:.0f}%',
            color='#94A3B8', fontsize=9, ha='center', monospace=True, zorder=100)
    
    return ax,

print('✅ Función de animación definida')

In [ ]:
# ── CREAR Y GUARDAR ANIMACIÓN ──
print('🎬 Creando animación...')
print('   (Esto puede tomar 2-5 minutos)\n')

anim = animation.FuncAnimation(
    fig, 
    animate,
    frames=len(datos_animacion),
    interval=100,  # 100ms entre frames = ~10 FPS
    repeat=True,
    blit=False
)

# Guardar como MP4
print('💾 Guardando como MP4 (300 DPI)...')
anim.save(
    'orbital_animation_fae.mp4',
    writer='ffmpeg',
    fps=10,
    dpi=100,
    bitrate=2000
)

plt.show()
print('✅ Animación guardada: orbital_animation_fae.mp4')
print('   Duración: ~6 segundos (60 minutos comprimidos)')

In [ ]:
# ── ANÁLISIS ESTADÍSTICO DE PASES ──
print('\n' + '='*70)
print('📊 ANÁLISIS DE PASES SATELITALES (60 minutos)')
print('='*70)

# Compilar todas las visibilidades por satélite-estación
pases = {}

for frame_idx, frame_data in enumerate(datos_animacion):
    t = frame_data['tiempo']
    
    for vis in frame_data['visibilidad']:
        key = (vis['sat_nombre'], vis['est_nombre'])
        
        if key not in pases:
            pases[key] = {
                'satellite': vis['sat_nombre'],
                'station': vis['est_nombre'],
                'constellation': vis['constelacion'],
                'rise_time': t,
                'set_time': t,
                'max_elevation': vis['elevacion'],
                'max_elev_time': t,
                'frame_count': 1
            }
        else:
            pases[key]['set_time'] = t
            pases[key]['frame_count'] += 1
            if vis['elevacion'] > pases[key]['max_elevation']:
                pases[key]['max_elevation'] = vis['elevacion']
                pases[key]['max_elev_time'] = t

# Convertir a lista y ordenar
pases_list = list(pases.values())
pases_list.sort(key=lambda x: x['max_elevation'], reverse=True)

print(f'\n🛰️  Total de pases detectados: {len(pases_list)}\n')

# Top 15 mejores pases
print('🏆 Top 15 mejores pases (por elevación máxima):\n')
print(f'{"Satélite":<15} {"Estación":<15} {"Elevación (°)":<12} {"Duración":<12} {"Hora Pico":<12}')
print('-' * 70)

for i, pase in enumerate(pases_list[:15]):
    duracion = (pase['set_time'] - pase['rise_time']).total_seconds() / 60
    hora_pico = pase['max_elev_time'].strftime('%H:%M:%S')
    
    print(f"{pase['satellite']:<15} {pase['station']:<15} {pase['max_elevation']:>10.1f}° {duracion:>10.1f} min  {hora_pico:<12}")

print('\n' + '='*70)

In [ ]:
# ── EXPORTAR A CSV ──
df_pases = pd.DataFrame([
    {
        'Satélite': p['satellite'],
        'Constelación': p['constellation'],
        'Estación': p['station'],
        'Hora Salida': p['rise_time'].strftime('%H:%M:%S'),
        'Hora Pico': p['max_elev_time'].strftime('%H:%M:%S'),
        'Hora Puesta': p['set_time'].strftime('%H:%M:%S'),
        'Elevación Máxima (°)': round(p['max_elevation'], 1),
        'Duración (min)': round((p['set_time'] - p['rise_time']).total_seconds() / 60, 1)
    }
    for p in pases_list
])

df_pases.to_csv('pases_satelitales_fae.csv', index=False)
print('✅ CSV exportado: pases_satelitales_fae.csv')
print(f'   Filas: {len(df_pases)}')

In [ ]:
# ── RESUMEN POR ESTACIÓN ──
print('\n' + '='*70)
print('📡 DISPONIBILIDAD POR ESTACIÓN (próxima hora)')
print('='*70 + '\n')

for estacion in estaciones:
    est_name = estacion['nombre']
    pases_est = [p for p in pases_list if p['station'] == est_name]
    
    print(f'📍 {est_name}')
    print(f'   Satélites visibles: {len(pases_est)}')
    print(f'   Total conexiones: {sum(p["frame_count"] for p in pases_est)} minutos')
    
    if pases_est:
        print(f'   Mejor pase: {pases_est[0]["satellite"]} @ {pases_est[0]["max_elevation"]:.1f}° ({pases_est[0]["max_elev_time"].strftime("%H:%M:%S")})')
        
        # Agrupar por constelación
        consts = {}
        for p in pases_est:
            const = p['constellation']
            consts[const] = consts.get(const, 0) + 1
        
        const_str = ', '.join([f"{c}({n})" for c, n in consts.items()])
        print(f'   Constelaciones: {const_str}')
    else:
        print('   ⚠️  Sin visibilidad en esta hora')
    
    print()

print('='*70)

In [ ]:
# ── RESUMEN FINAL ──
print('\n' + '🎬 '*10)
print('ANIMACIÓN COMPLETADA CON ÉXITO'.center(70))
print('🎬 '*10 + '\n')

print('📁 Archivos generados:')
print('   ✅ orbital_animation_fae.mp4 (~15-30 MB)')
print('   ✅ pases_satelitales_fae.csv')

print('\n📊 Estadísticas:')
print(f'   • Duración simulada: 60 minutos')
print(f'   • Duración video: ~6 segundos (comprimido)')
print(f'   • Frames: {len(datos_animacion)}')
print(f'   • Total pases: {len(pases_list)}')
print(f'   • Satélites activos: {len(set(p["satellite"] for p in pases_list))}')
print(f'   • Estaciones con cobertura: {len(set(p["station"] for p in pases_list))}')

print('\n🎓 Cómo usar:')
print('   1. Descarga orbital_animation_fae.mp4')
print('   2. Abre en cualquier reproductor de video (VLC, etc)')
print('   3. Observa cómo los satélites orbitan Ecuador en tiempo comprimido')
print('   4. Consulta pases_satelitales_fae.csv para detalles exactos')

print('\n' + '='*70)